# PUBG: Khai phá hành vi người chơi
Bài cuối kỳ AI và Khoa học Dữ liệu, thiết kế 5.11, chạy trực tiếp trên Kaggle. Một notebook chứa toàn quy trình và chế độ đọc kết quả.

Ba câu hỏi: RQ1 quan hệ hành vi với sống sót/thứ hạng; RQ2 nhóm hành vi theo Solo/Duo/Squad; RQ3 hồi quy hậu trận và giá trị bổ sung của Combat Timing. Không diễn giải nhân quả hoặc dự báo trước trận từ các biến đã biết sau trận.

Đọc các phần theo thứ tự; kiểm **scope**, **N**, **đơn vị**, **missing**, **status** trước kết luận. Dữ liệu kiểm thử không phải kết quả PUBG thật. Quy trình C1 (KDD) kết hợp C2 (tiền xử lý), C6 (hồi quy), C7 (phân cụm); C4 hỗ trợ đánh giá.

Mục lục: [S00 Cấu hình](#S00), [S01 Nguồn](#S01), [S02 Chất lượng/split](#S02), [S03 Features](#S03), [S04 Timing](#S04), [S05 EDA](#S05), [S06 RQ1](#S06), [S07 RQ2](#S07), [S08 RQ3](#S08), [S09 Sai số](#S09), [S10 Khóa kết quả](#S10), [S11 Đọc kết quả](#S11), [S12 Bàn giao](#S12).

Chế độ tự động được duyệt ngày 17/10/2026: Run All dùng quy tắc đăng ký trước, mỗi mục có OPTIONS để ghi đè. Không tự suy đơn vị hoặc ý nghĩa nguồn. Kiểm tài nguyên vẫn có thể dừng; không tự lấy mẫu. Không cần Save Version sau từng mục. Các output kiểm thử lịch sử đã được bỏ khỏi bản code mới; chưa có kết quả chạy tự động trên PUBG thật.

Ngày 18/10/2026, người dùng duyệt giả định giây cho nguồn PUBG Version 3 sau đối chiếu mẫu ZIP. Trạng thái assumed_seconds không phải verified. Bảng, hình, mô hình và release phải công bố giới hạn; không đổi thang đo hoặc bỏ các dòng lệch chỉ vì giả định này.

Hai GPU: các K ứng viên và fit per-mode ở S07, các hồi quy độc lập ở S08 chạy trong tiến trình cô lập GPU0/GPU1. Không chia nhỏ một fit, không cộng VRAM; giữ toàn eligible data và thuật toán. Chọn T4 x2; S00 kiểm đủ GPU và smoke48dòng/GPU trước ingest.

<a id="S00"></a>
## S00. Cấu hình và kiểm tra trước khi chạy
Mục tiêu: xác nhận đúng dự án mới, môi trường, phạm vi và nơi lưu. `full` dùng toàn eligible cohort; `development` là kiểm thử; `report_only` chỉ đọc một release cụ thể.

Dùng Add Input gắn nguồn Kaggle đã pin version và gói code mới. `CODE_INPUT` là folder code chỉ đọc; `PUBG_PROJECT_ROOT` là working root. Không copy raw vào working hoặc dùng checkpoint dự án cũ. `RESUME_INPUT` chỉ chọn output/version đã lưu, có handoff manifest.

Bản upload chính thức này đã cấu hình `full`, cho phép xử lý toàn bộ dữ liệu nguồn PUBG Version 3 theo yêu cầu ngày 18/10/2026, từ S01 đến S10. Cell tham số gán trực tiếp, không giữ lại `False` hoặc `pending` từ RAM cũ. Kiểm đúng hai Input và pin Version 3 trước Run All; các kiểm tra nguồn, đơn vị và tài nguyên vẫn giữ nguyên.

START_STAGE/END_STAGE giới hạn phần tính toán. Runtime mới luôn chạy S00 trước; checkpoint đã completed compatible mới reuse. Nếu có quyết định null, xem diagnostics và dừng, chốt config rồi chạy lại S00 và section liên quan.

CPU dùng SQL/EDA/metrics; KMeans/OLS dùng backend device đã khai báo. GPU T4 không tăng RAM CPU. Bộ phụ thuộc GPU cài riêng theo CUDA của runtime, không fallback ngầm.

Bản5.4 mặc định rq2/rq3.gpu_ids=[0,1]. RUN_GPU_SMOKE=True kiểm KMeans/OLS và reload nhỏ trên cả hai GPU bằng CUDA_VISIBLE_DEVICES riêng cho từng tiến trình. Thiếu một GPU thì dừng, không âm thầm chạy một GPU/CPU. Tổng hostRAM và VRAM từng GPU vẫn phải đủ.

In [ ]:
PUBG_STORAGE_MODE = "local"
PUBG_PROJECT_ROOT = "/kaggle/working/PUBG_Coursework_v511"
PUBG_REQUIRE_EXISTING_PROJECT = True
PUBG_BATCH_ROWS = 50000
EXECUTION_MODE = "full"
CODE_INPUT = "/kaggle/input/datasets/quangthinng/pubg-coursework-code"
SOURCE_DIRECTORY = "/kaggle/input/datasets/skihikingkevin/pubg-match-deaths"
SOURCE_VERSION = 3
SOURCE_LICENSE = "CC0: Public Domain"
ALLOW_DATA_PROCESSING = True
ALLOW_FULL_DATA = True
START_STAGE = "S01"
END_STAGE = "S10"
RESUME_INPUT = ""
RESUME_VERSION = ""
S01_IMPORT_ROOT = ""  # Chỉ nhập S01 đã kiểm từ working5.10, không phải RESUME_INPUT.
CLEANUP_IMPORTED_510 = False  # Chỉ True khi đồng ý bỏ cleaned S02 dở và các tên liên kết S01 cũ.
RELEASE_MANIFEST = ""
SAVED_OUTPUT_ROOT = ""
SAVED_VERSION = ""
RUN_GPU_SMOKE = True
INSTALL_DEPENDENCIES = False
PUBG_WRITER_ID = ""
PUBG_FORCE_STAGE_TAKEOVER = False
HANDOFF_EVERY_STAGE = False
SOURCE_MODE_EVIDENCE = ""  # Kế thừa audit mẫu trong config5.3.
SOURCE_TIME_UNIT_STATUS = "assumed_seconds"
SOURCE_TIME_UNIT_EVIDENCE = ""  # Kế thừa evidence/giới hạn trong config5.3.
SOURCE_TIME_UNIT_ASSUMPTION_APPROVED = True
SOURCE_TIME_UNIT_ASSUMPTION_SCOPE = "skihikingkevin/pubg-match-deaths/versions/3"


In [ ]:
import sys, subprocess
from pathlib import Path
sys.dont_write_bytecode = True
if EXECUTION_MODE not in {"development","full","report_only"} or PUBG_STORAGE_MODE != "local":
    raise ValueError("Dùng local working trên Kaggle; execution mode phải hợp lệ")
if PUBG_BATCH_ROWS != 50000 or not PUBG_REQUIRE_EXISTING_PROJECT:
    raise ValueError("Giữ batch50000 và require-existing=True")
if EXECUTION_MODE == "report_only" and (CODE_INPUT or RESUME_INPUT or INSTALL_DEPENDENCIES or RUN_GPU_SMOKE or S01_IMPORT_ROOT or CLEANUP_IMPORTED_510):
    raise ValueError("report_only không copy/restore/install/smoke; chuẩn bị code/environment trước")
if EXECUTION_MODE != "report_only":
    if not ALLOW_DATA_PROCESSING:
        raise ValueError("Chưa cho phép đọc/xử lý nguồn. Kiểm path/version và phạm vi nhỏ trước bật ALLOW_DATA_PROCESSING")
    if EXECUTION_MODE == "full" and not ALLOW_FULL_DATA:
        raise ValueError("Chưa cho phép full-data; không suy development là dữ liệu nhỏ")
    if START_STAGE not in [f"S{i:02d}" for i in range(1,11)] or END_STAGE not in [f"S{i:02d}" for i in range(1,11)] or START_STAGE > END_STAGE:
        raise ValueError("Stage range phải S01..S10 và START<=END")
project_root = Path(PUBG_PROJECT_ROOT).resolve()
import_root = Path(RESUME_INPUT or CODE_INPUT).resolve() if (RESUME_INPUT or CODE_INPUT) else project_root
if not (import_root/"src/io.py").is_file() or not (import_root/"configs/project.yaml").is_file():
    raise FileNotFoundError("Chọn CODE_INPUT/RESUME_INPUT hoặc root bản mới có src/io.py/configs/project.yaml")
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable,"-m","pip","install","-r",str(import_root/"requirements.txt")])
    raise RuntimeError("Đã cài dependencies. Restart kernel, đặt INSTALL_DEPENDENCIES=False rồi chạy lại S00 để tránh trộn thư viện cũ trong RAM")
import runpy
guard = import_root / 'src/bootstrap.py'
if not guard.is_file():
    raise RuntimeError('Code Dataset cũ: cập nhật gói code 5.11 trước Run All; không xóa working còn S01 cần nhập')
bundle = runpy.run_path(str(guard))['check_bundle'](import_root, '5.11-normalized-storage', '4e3f9a191e589c963727c12bbf00ec656ee6bf17de0621ca6a6f520b3641dd3c')
print(bundle)
sys.path.insert(0,str(import_root))
from IPython.display import display, Image
import pandas as pd
import yaml
from src.io import Context, PendingDecision, read_json, load_config, stage_options, time_unit_info
from src.kaggle_runtime import prepare_project, preflight, gpu_smoke, prepare_handoff, verify_handoff, restore_handoff, resource_plan, import_s01_510, cleanup_imported_510
from src.preprocessing import ingest, clean_split
from src.features import base_features, combat_timing
from src.analysis import eda, relationship, clustering
from src.modeling import prediction, comparisons
from src.reporting import finalize, display_release
if EXECUTION_MODE != "report_only":
    if S01_IMPORT_ROOT:
        if RESUME_INPUT or not CODE_INPUT:
            raise ValueError("Import S01 cần CODE_INPUT mới và RESUME_INPUT trống")
        project_root = import_s01_510(S01_IMPORT_ROOT,CODE_INPUT,project_root,EXECUTION_MODE,SOURCE_DIRECTORY,SOURCE_VERSION,SOURCE_LICENSE)
        if CLEANUP_IMPORTED_510:
            print(cleanup_imported_510(S01_IMPORT_ROOT,project_root,EXECUTION_MODE,confirm=True))
    elif CLEANUP_IMPORTED_510:
        raise ValueError("Không cleanup nếu chưa chọn S01_IMPORT_ROOT")
    elif RESUME_INPUT:
        project_root = restore_handoff(RESUME_INPUT,project_root,EXECUTION_MODE,SOURCE_DIRECTORY,RESUME_VERSION)
    elif CODE_INPUT:
        project_root = prepare_project(CODE_INPUT,project_root)
    cfg = load_config(project_root)
    changed = False
    for key,value in (("directory",SOURCE_DIRECTORY),("version",SOURCE_VERSION),("license",SOURCE_LICENSE)):
        if value not in (None,"") and cfg["source"].get(key) != value:
            cfg["source"][key] = value; changed = True
    for section, key, value in (
        ("preprocessing", "mode_evidence", SOURCE_MODE_EVIDENCE),
        ("timing", "unit_status", SOURCE_TIME_UNIT_STATUS),
        ("timing", "unit_evidence", SOURCE_TIME_UNIT_EVIDENCE),
        ("timing", "unit_assumption_approved", SOURCE_TIME_UNIT_ASSUMPTION_APPROVED),
        ("timing", "unit_assumption_scope", SOURCE_TIME_UNIT_ASSUMPTION_SCOPE),
    ):
        if value not in (None, "") and cfg[section].get(key) != value:
            cfg[section][key] = value
            changed = True
    if changed:
        (project_root/"configs/project.yaml").write_text(yaml.safe_dump(cfg,allow_unicode=True,sort_keys=False),encoding="utf-8")
    progress_view = display(pd.DataFrame([{"step":"Sẵn sàng; chưa scan nguồn"}]),display_id=True)
    def show_progress(event):
        progress_view.update(pd.DataFrame([event]))
    ctx = Context(project_root,EXECUTION_MODE,PUBG_WRITER_ID,PUBG_FORCE_STAGE_TAKEOVER,progress=show_progress)
    display(ctx.environment())
    display(pd.DataFrame([{"key":k,"value":str(v)} for k,v in preflight(ctx,START_STAGE,END_STAGE,kaggle=Path("/kaggle/input").exists()).items()]))
    display(pd.DataFrame([time_unit_info(ctx.cfg)]))
    if time_unit_info(ctx.cfg)["status"] == "assumed_seconds":
        print("GIỚI HẠN ĐƠN VỊ: " + time_unit_info(ctx.cfg)["limitation"])
    if RUN_GPU_SMOKE:
        display(pd.DataFrame([gpu_smoke(ctx)]))
    display(pd.DataFrame([{"key":k,"value":str(v)} for k,v in ctx.cfg.items()]))
    display(pd.DataFrame([{"stage":k,"status":v.get("status"),"scope":v.get("scope")} for k,v in ctx.manifest().items()]))
else:
    load_config(project_root)
    if not RELEASE_MANIFEST:
        raise ValueError("report_only yêu cầu RELEASE_MANIFEST cụ thể, không lấy latest")
    print("Chỉ đọc release; không tạo Context hoặc pipeline stage")

Cách đọc: source version/path phải đúng Input đã pin. RAM/disk là số runtime, không quota GPU hoặc giới hạn saved output. Working checkpoint khác Kaggle Version đã lưu; chỉ verify_handoff trên output gắn lại mới có read-back evidence. Tiến độ là số đã xử lý tạm, không thống kê toàn nguồn. Cài dependencies trước report_only, giữ environment giống nhau để reload; chưa biết license/units/K/G4 thì không chốt thay.

In [ ]:
STAGES = {"S01":ingest,"S02":clean_split,"S03":base_features,"S04":combat_timing,
          "S05":eda,"S06":relationship,"S07":clustering,"S08":prediction,"S09":comparisons,"S10":finalize}
def show_stage(stage):
    record = ctx.manifest().get(stage,{})
    paths = [ctx.root/a["path"] for a in record.get("artifacts",[])]
    if record.get("status") != "completed":
        paths = list(ctx.created)
    for path in paths:
        if path.suffix==".csv" and "assignments" not in path.name:
            print(path.name); display(pd.read_csv(path, nrows=20))
        elif path.suffix==".png":
            display(Image(filename=str(path)))
    display(pd.DataFrame([{"stage":stage,"status":record.get("status"),"reason":record.get("reason",""),
                           "artifacts":len(record.get("artifacts",[])),"checkpoint":str(ctx.manifest_path)}]))
def run_section(stage, options=None):
    if EXECUTION_MODE=="report_only":return
    if not START_STAGE <= stage <= END_STAGE:
        print(stage+": ngoài khoảng thực thi; không tính lại");return
    stage_options(ctx, stage, options)
    preflight(ctx, stage, stage, kaggle=Path("/kaggle/input").exists())
    if stage == 'S05' and END_STAGE >= 'S07':
        plan = resource_plan(ctx, stage, END_STAGE)
        display(pd.DataFrame(plan['checks']))
        print('RAM budget GiB:', plan['host_budget_GiB'], ';', plan['limitation'])
        if plan['feasible'] is False:
            raise MemoryError('Dữ liệu hợp lệ vượt RAM dự toán S07/S08; cần máy nhiều RAM hơn. Không giảm dữ liệu hoặc đổi thuật toán.')
    try:
        ctx.run(stage,STAGES[stage])
    finally:
        show_stage(stage)
        if HANDOFF_EVERY_STAGE or stage == END_STAGE:
            try:
                path = prepare_handoff(ctx)
                print("Gói bàn giao runtime:", path, "; chưa xác minh Kaggle Version đã lưu.")
            except (RuntimeError, ValueError, MemoryError, OSError) as error:
                print("Chưa tạo được gói bàn giao:", str(error))
        else:
            print("Checkpoint runtime đã ghi; không cần Save Version giữa các stage. Chưa có bản lưu bền vững.")


<a id="S01"></a>
## S01. Nguồn, schema và đối soát

Câu hỏi kiểm tra: Rows đọc và rows Parquet phải khớp; thiếu required hoặc alias collision dừng trước downstream. Metadata source/version/license/download_date chưa biết giữ unknown, không dùng mtime.

Input/provenance: stage dependencies được kiểm từ manifest/checksum; phạm vi lấy từ S00, không lấy latest theo tên file.

Phương pháp và lý do: Một nhóm aggregate có player-match; một nhóm event có killer/victim/time. Discovery mọi shard, không hard-code hai file. Header required/alias phải đúng; parse errors khác missing gốc. CSV được đọc theo luồng, Parquet từng shard có lineage để phát hiện conflict xuyên shard. Các shard Parquet chỉ là file tạm: sau khi aggregate.parquet hoặc event.parquet đã được giải mã, kiểm số dòng và publish/read-back thành công, chúng được dọn khỏi thư mục tạm riêng của lượt chạy. Không giữ thư mục typed trùng dữ liệu, không bỏ raw columns hoặc lấy mẫu.

In [ ]:
# None: kế thừa cấu hình; sửa giá trị trước khi chạy cell này.
S01_OPTIONS = {
    'directory': None,
    'version': None,
    'license': None
}
run_section("S01", S01_OPTIONS)

Cách đọc kết quả: Inventory bar theo shard và parse/missing cho thấy dữ liệu đi vào; top20 chỉ preview, CSV nguồn đủ hàng. Raw mới giữ bất biến. source.directory null phải nhập sau xác định nguồn.

Hạn chế và bàn giao: Inventory/hash/schema/parse trong outputs; aggregate.parquet và event.parquet trong data cho consumer S02/S04. ingest_storage.csv ghi số dòng, số shard, byte file tạm đã dọn và byte bản gộp; temporary_files_remaining phải bằng 0 sau thành công. Đây không phải số đo đỉnh dung lượng toàn quy trình. Đơn vị chưa xác minh sẽ chặn S04.

Nếu bảng/hình chưa sinh hoặc stage pending, chưa có kết luận nghiên cứu. Hãy đọc reason, sửa đúng config/source rồi chạy lại S00 và phần này.

<a id="S02"></a>
## S02. Chất lượng, roster và split
Bản 5.6: bảng trung gian ở database tạm trên đĩa; thông tin đội, roster, ngày và chế độ được tổng hợp riêng trước khi ghép. Không giảm dữ liệu hoặc tăng giới hạn DuckDB. s02_execution.json ghi cách thực thi.

Câu hỏi kiểm tra: Đối soát từng bước: before=removed+after. Một match chỉ một split; chronology Grade B không xé ngày, A không xé timestamp ties, C chia group match. Tỷ lệ70/15/15 thực đạt có thể lệch do nguyên block.

Input/provenance: stage dependencies được kiểm từ manifest/checksum; phạm vi lấy từ S00, không lấy latest theo tên file.

Phương pháp và lý do: Exact duplicates được gộp xuyên shard theo nội dung nguồn; cùng identity khác nội dung giữ conflict audit và loại khỏi cohort theo quy tắc. Flags lỗi chồng lặp khác ledger loại tuần tự. Metadata roster trước lọc theo target.

Tự động: dùng OPTIONS trong cell thực thi; None kế thừa config. Chế độ manual vẫn giữ cổng chờ quyết định. Quyết định tự động được lưu vào manifest trước khi dùng; không dùng test để chọn tham số.

In [ ]:
# None: kế thừa cấu hình; sửa giá trị trước khi chạy cell này.
S02_OPTIONS = {
    'decision_mode': None,
    'mode_evidence': None,
    'chronology_grade': None,
    'chronology_reason': None,
    'chronology_evidence': None
}
run_section("S02", S02_OPTIONS)

Cách đọc kết quả: Removal plot cần đọc tổng dòng theo bước, không cộng flags. Roster N_teams/placement consistency quyết định valid target. mode mapping có evidence, Other riêng. Không dùng test để ra quyết định EDA/model.

Hạn chế và bàn giao: Cleaned/metadata/split mới cùng split manifest; consumer S03. Timestamp có giây chưa đủ Grade A, tên người chơi không phải ID bất biến.

Nếu bảng/hình chưa sinh hoặc stage pending, chưa có kết luận nghiên cứu. Hãy đọc reason, sửa đúng config/source rồi chạy lại S00 và phần này.

<a id="S03"></a>
## S03. Feature cơ sở và target

Câu hỏi kiểm tra: Normalized placement y=1-(rank-1)/(N_teams-1), winner1/last0. Cần roster đáng tin và N_teams>1; rank ngoài miền không clip. Survival valid và placement valid tách task.

Input/provenance: stage dependencies được kiểm từ manifest/checksum; phạm vi lấy từ S00, không lấy latest theo tên file.

Phương pháp và lý do: Các biến raw/derived theo nhóm Combat/Movement/Support. Damage/kill, walk/total và assists/(kills+assists) mẫu số0 là missing, không epsilon. Metadata đội không tính lại trên model cohort.

Công thức cần kiểm chứng:

$$
D_{\mathrm{total}}=D_{\mathrm{walk}}+D_{\mathrm{ride}}
$$

$$
r_{\mathrm{walk}}=\frac{D_{\mathrm{walk}}}{D_{\mathrm{total}}}
$$

$$
r_{\mathrm{assist}}=\frac{\mathrm{assists}}{\mathrm{kills}+\mathrm{assists}}.
$$

$$
y_{\mathrm{placement}}=1-\frac{\mathrm{rank}-1}{N_{\mathrm{teams}}-1}.
$$

Ví dụ roster có 4 đội: rank 1 cho 1, rank 4 cho 0. Điều kiện mẫu số và roster áp dụng trước tính; mẫu số 0 là missing có lý do. Không clip để che lỗi.

In [ ]:
# None: kế thừa cấu hình; sửa giá trị trước khi chạy cell này.
S03_OPTIONS = {}
run_section("S03", S03_OPTIONS)

Bản 5.7: S03 dùng database tạm trên đĩa, JOIN metadata chỉ đọc cột cần thiết, kiểm khóa và bảo toàn số dòng. Validation tổng hợp mọi cột trong một lượt scan; công thức, kiểu và quy tắc missing không đổi.

Cách đọc kết quả: Dictionary/validation N/valid/missing/min/max giúp phát hiện sai phép chia. Ví dụ tính tay chỉ minh họa; đối chiếu với actual rows. Rate/velocity chứa survival không dùng S1 hoặc RQ1 primary-survival.

Hạn chế và bàn giao: Base Parquet giữ identity/lineage/targets/flags. Consumer S04; ghi đơn vị dựa nguồn, không gán verified từ dự án cũ.

Nếu bảng/hình chưa sinh hoặc stage pending, chưa có kết luận nghiên cứu. Hãy đọc reason, sửa đúng config/source rồi chạy lại S00 và phần này.

<a id="S04"></a>
## S04. Combat Timing và hợp nhất

Câu hỏi kiểm tra: Proxy duration=max survival trong match, không official duration. Quy ước min60 giây, Early[0,1/3), Mid[1/3,2/3), Late[2/3,1]. Absolute timing hợp lệ giữ ngay cả phase invalid/out-of-range; không clip.

Input/provenance: stage dependencies được kiểm từ manifest/checksum; phạm vi lấy từ S00, không lấy latest theo tên file.

Phương pháp và lý do: Event credits named non-self killer, valid match/time. Có thể gồm same-team/unknown-team, không enemy-only. First=min; avg=sum/time_count qua toàn shard, không trung bình chunk means.

Bản 5.7: ghi events/metadata/base một lượt vào 16 phần theo SHA256(match_id). Mọi dòng cùng trận ở cùng phần, dedup vẫn theo toàn nội dung gốc, không theo hash. Tính và JOIN từng phần trên đĩa, kiểm số dòng rồi mới ghép ra player_match.parquet. timing_partition_audit.csv và s04_execution.json ghi số dòng/cách chạy. Đổi lại cần thêm đĩa tạm và I/O, không hứa nhanh hơn. Chưa có resume từng phần hoặc tự tách phần lệch lớn; lỗi sẽ chạy lại S04, không đánh completed.

Tự động: dùng OPTIONS trong cell thực thi; None kế thừa config. Chế độ manual vẫn giữ cổng chờ quyết định. Quyết định tự động được lưu vào manifest trước khi dùng; không dùng test để chọn tham số.

Đơn vị: verified cần evidence xác minh. assumed_seconds chỉ được chấp nhận khi có evidence, approval=True và scope đúng dataset/version. Cấu hình bản 5.3 lưu chấp thuận ngày 18/10/2026 cho nguồn PUBG Version 3. Đây là giả định, không phải thông tin đã được tác giả nguồn xác nhận. Nếu S00 còn dòng SOURCE_TIME_UNIT_STATUS="pending" từ cấu hình cũ, cần đổi rõ thành "assumed_seconds" hoặc để trống để kế thừa config mới, không đổi thành verified.

Định nghĩa thời gian:

$$
T_m=\max_{i\in m}s_i,\qquad u_e=\frac{t_e}{T_m}
$$

$$
\overline{t}_{p,m}=\frac{\sum_e t_e}{n_{p,m}}.
$$

Qua các batch $b$, cộng $\sum_b \mathrm{sum}_b$ và $\sum_b n_b$ trước chia. Ví dụ một batch có 10, batch kia có 20/30/40: mean đúng 25, không phải mean của 10 và 30 là 20. First là global min 10.

Early $0\le u<1/3$, Mid $1/3\le u<2/3$, Late $2/3\le u\le1$. Với $T_m=300$, event 100 là Mid, 200 và 300 là Late. Đây là pha tương đối theo proxy hậu trận, không phải phút thực tế cố định.

In [ ]:
# None: kế thừa cấu hình; sửa giá trị trước khi chạy cell này.
S04_OPTIONS = {
    'partitions': None,  # Mặc định16; nếu đổi làm stale S03, chạy lại S03 rồi S04.
    'unit_status': None,
    'unit_evidence': None,
    'unit_assumption_approved': None,
    'unit_assumption_scope': None,
    'minimum_duration': None
}
run_section("S04", S04_OPTIONS)

Cách đọc kết quả: Đọc event audit/coverage/discrepancy trước hiểu feature. No-kill confirmed khác aggregate-kills>0 nhưng event missing; missing ratio không phải0. Phân bố phase có phase_valid denominator. LEFT JOIN không tăng số player rows.

Hạn chế và bàn giao: Player_match Parquet và full discrepancy Parquet mới; consumer S05-S09. Cần timing.unit_status/evidence từ nguồn/consistency mới; proxy chuẩn hóa tạo coupling với survival.

Nếu bảng/hình chưa sinh hoặc stage pending, chưa có kết luận nghiên cứu. Hãy đọc reason, sửa đúng config/source rồi chạy lại S00 và phần này.

Đơn vị: verified cần evidence xác minh. assumed_seconds chỉ được chấp nhận khi có evidence, approval=True và scope đúng dataset/version. Cấu hình bản 5.3 lưu chấp thuận ngày 18/10/2026 cho nguồn PUBG Version 3. Đây là giả định, không phải thông tin đã được tác giả nguồn xác nhận. Nếu S00 còn dòng SOURCE_TIME_UNIT_STATUS="pending" từ cấu hình cũ, cần đổi rõ thành "assumed_seconds" hoặc để trống để kế thừa config mới, không đổi thành verified.

<a id="S05"></a>
## S05. EDA bám thống kê và tiền xử lý C2

Câu hỏi kiểm tra: Giữ cực trị hợp lệ; boxplot/hist không tự tạo luật xóa. Log-transform hoặc giữ nguyên cần transform_reason. Mode comparisons dùng hành vi cho representation, không outcome chọn cụm.

Input/provenance: stage dependencies được kiểm từ manifest/checksum; phạm vi lấy từ S00, không lấy latest theo tên file.

Phương pháp và lý do: Dùng train+validation cho biến đổi và thiết kế, không mở final test. Thống kê N/median/mean/std/quantiles/skew/missing/zero toàn scope; SQL cho dữ liệu lớn, không collect raw thành pandas.

Bản 5.8: một lượt quét lấy các thống kê có trạng thái nhỏ. Median/q25/q75 vẫn chính xác: sắp xếp từng cột toàn cục trên database đĩa, lấy vị trí cần thiết, dùng kernel nội suy Type-7 gốc trên tối đa hai đầu mút cho mỗi phân vị. Không dùng trung vị từng lô hoặc phân vị xấp xỉ. Histogram dùng lại min/max đã tính; giữ điều kiện valid_survival riêng cho target survival. s05_execution.json ghi cách thực thi, không phải bản lưu bền vững. Đổi lại cần đĩa tạm và đọc/ghi thêm.

Tự động: dùng OPTIONS trong cell thực thi; None kế thừa config. Chế độ manual vẫn giữ cổng chờ quyết định. Quyết định tự động được lưu vào manifest trước khi dùng; không dùng test để chọn tham số.

In [ ]:
# None: kế thừa cấu hình; sửa giá trị trước khi chạy cell này.
S05_OPTIONS = {
    'decision_mode': None,
    'log_columns': None,
    'transform_reason': None
}
run_section("S05", S05_OPTIONS)

Cách đọc kết quả: Histograms có bins/units, missing khác zero. Nhìn lệch/phân tán/mode trước model. Dữ liệu người chơi lặp, outcome đội lặp và cơ hội tích lũy hành vi khi sống lâu đều hạn chế suy luận.

Hạn chế và bàn giao: EDA tables/plots/scope record; consumer S06/S07/S08. Nếu chưa chốt transforms lưu diagnostics rồi dừng; fixture không chốt thay dữ liệu thật.

Nếu bảng/hình chưa sinh hoặc stage pending, chưa có kết luận nghiên cứu. Hãy đọc reason, sửa đúng config/source rồi chạy lại S00 và phần này.

<a id="S06"></a>
## S06. RQ1: quan hệ hành vi và outcome

Câu hỏi kiểm tra: Pair N phải có cho mỗi feature-target-mode. Constant/insufficient là undefined/NA, không0. Timing phase dùng duration proxy không primary với survival. P nhỏ khi N lớn không chứng minh quan hệ mạnh.

Input/provenance: stage dependencies được kiểm từ manifest/checksum; phạm vi lấy từ S00, không lấy latest theo tên file.

Phương pháp và lý do: Pearson đo tuyến tính, Spearman đo quan hệ đơn điệu theo ranks; ranks/ties trên valid pairs toàn scope. Overall và từng mode riêng, giữ dấu hệ số.

Bản 5.8: mỗi feature-target-mode có bảng cặp hữu hạn riêng trên đĩa. Đếm tần số giá trị toàn cặp và cộng dồn để tạo average ranks, rồi JOIN hạng x/y theo từng bước, kiểm bảo toàn N. Với giá trị v, hạng trung bình là $r(v)=N_{<v}+(N_v+1)/2$. Không tái dùng rank của cặp khác hoặc trung bình tương quan từng phần. s06_execution.json ghi phương pháp; constant/empty vẫn undefined, không thay bằng0.

Hai hệ số:

$$
r=\frac{\sum_i(x_i-\bar x)(y_i-\bar y)}
{\sqrt{\sum_i(x_i-\bar x)^2\sum_i(y_i-\bar y)^2}},
\qquad \rho=\mathrm{corr}(\mathrm{rank}(x),\mathrm{rank}(y)).
$$

Ties dùng average ranks trên đúng tập valid pairs. Mỗi cặp có N riêng; constant hoặc ít quan sát không có hệ số đáng diễn giải. Không tính riêng từng batch rồi trung bình hệ số.

In [ ]:
# None: kế thừa cấu hình; sửa giá trị trước khi chạy cell này.
S06_OPTIONS = {}
run_section("S06", S06_OPTIONS)

Cách đọc kết quả: Signed bars so Pearson/Spearman; xem khác mode và valid flags. Không kết luận walking gây survival hoặc early kills gây thắng. Mỗi phát biểu cần feature/target/scope/N/hệ số/giới hạn.

Hạn chế và bàn giao: rq1_relationships.csv và hình nguồn; consumer S10/S11. EDA quan hệ lựa chọn chưa dùng test; full description không được quay lại tune.

Nếu bảng/hình chưa sinh hoặc stage pending, chưa có kết luận nghiên cứu. Hãy đọc reason, sửa đúng config/source rồi chạy lại S00 và phần này.

<a id="S07"></a>
## S07. RQ2: KMeans riêng theo mode, bám C7

Câu hỏi kiểm tra: Chốt minimum_games từ retention/coverage, K từ inertia/silhouette/sizes/stability qua seed trên cùng reference profiles. Imputation/scaling học development; full descriptive fit chỉ sau khóa. Outcomes không input hoặc tiêu chí chọn K.

Input/provenance: stage dependencies được kiểm từ manifest/checksum; phạm vi lấy từ S00, không lấy latest theo tên file.

Phương pháp và lý do: Player-mode profile Design3: mean/std combat, movement/support và timing ratios. Conditional means phải có valid counts; mean ratio khác pooled ratio. Singleton std không phải0. Games_played filter reliability, không predictor core.

Tự động: dùng OPTIONS trong cell thực thi; None kế thừa config. Chế độ manual vẫn giữ cổng chờ quyết định. Quyết định tự động được lưu vào manifest trước khi dùng; không dùng test để chọn tham số.

StandardScaler và mục tiêu KMeans:

$$
z_{ij}=\frac{x_{ij}-\mu_j}{\sigma_j}
$$

$$
\min_{\{C_k,\mu_k\}}\sum_{k=1}^{K}\sum_{i\in C_k}\|z_i-\mu_k\|_2^2.
$$

Chuẩn hóa tránh khoảng cách bị biến có thang đo lớn chi phối; KMeans ưu tiên cụm gần dạng cầu và nhạy ngoại lệ/khởi tạo. Inertia giảm theo K nên cần silhouette/kích thước/stability và diễn giải cùng nhau.

Mean phase ratio là $\frac{1}{n_{\mathrm{valid\ matches}}}\sum_m r_m$. Hai trận 1/1 và 0/3 cho mean 0.5; pooled ratio 1/4=0.25 là thống kê khác. Độ lệch chuẩn mẫu dùng $n-1$; singleton là NA, không zero variance.

In [ ]:
# None: kế thừa cấu hình; sửa giá trị trước khi chạy cell này.
S07_OPTIONS = {
    'decision_mode': None,
    'gpu_ids': None,  # Kế thừa [0, 1]; không đổi khi stage đang chạy.
    'minimum_games': None,
    'candidate_k': None,
    'clusters': None
}
run_section("S07", S07_OPTIONS)

Cách đọc kết quả: Retention/K plots nêu diagnostic sample n/seed/rule; full fit không lấy mẫu. Standardized centers heatmap/cluster sizes giúp đặt tên hành vi. C5 outcomes chỉ sau assignments khóa; cluster IDs không so trực tiếp giữa modes.

Hạn chế và bàn giao: Lưu imputer/scaler/features/model/assignments/centers và receipts mỗi mode. Reload labels phải khớp. Null hoặc stale receipt dừng; GPU routing không fallback CPU. Timing proxy là giới hạn representation.

Nếu bảng/hình chưa sinh hoặc stage pending, chưa có kết luận nghiên cứu. Hãy đọc reason, sửa đúng config/source rồi chạy lại S00 và phần này.

Bản5.9: tổng hợp profile trên database đĩa, giữ nhãn seed đầu thay vì toàn bộ các seed, đọc metadata từng mode và reload theo lô50000. Davies-Bouldin tính khoảng cách hiệu tọa độ trực tiếp để tránh cancellation, không đổi định nghĩa hoặc quy tắc chọn K. Collect profile và full KMeans vẫn cần RAM/VRAM; không fit theo từng phần.

<a id="S08"></a>
## S08. RQ3: baseline và hồi quy tuyến tính C6

Câu hỏi kiểm tra: Train-only preprocessing và fit. Validation chọn thiết kế; G4 cần approval/reason/hash trước final test. T0/T1 same rows/target/split/model/preprocessing policy; mỗi nhánh fit trên train đúng cột. Không dùng actual để tạo prediction.

Input/provenance: stage dependencies được kiểm từ manifest/checksum; phạm vi lấy từ S00, không lấy latest theo tên file.

Phương pháp và lý do: S1 hồi quy survival với features không chứa target/descendants; P2 placement bỏ survival trực tiếp, vẫn có timing proxy coupling. Train mean/median và Linear Regression; T0 bỏ timing, T1 reuse P2 khi signature/cohort khớp.

Tự động: dùng OPTIONS trong cell thực thi; None kế thừa config. Chế độ manual vẫn giữ cổng chờ quyết định. Quyết định tự động được lưu vào manifest trước khi dùng; không dùng test để chọn tham số.

Mô hình hồi quy:

$$
\hat y_i=\beta_0+\sum_{j=1}^{p}\beta_j z_{ij}
$$

$$
\hat\beta=\arg\min_{\beta}\sum_{i\in \mathrm{train}}(y_i-\hat y_i)^2.
$$

Baseline mean/median chỉ học $y_{\mathrm{train}}$. Hệ số không là tác động nhân quả, đặc biệt khi predictors tương quan.

$$
MAE=\frac{1}{n}\sum_i|y_i-\hat y_i|
$$

$$
RMSE=\sqrt{\frac{1}{n}\sum_i(y_i-\hat y_i)^2}
$$

$$
R^2=1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y)^2}.
$$

R² có thể âm; target hằng hoặc n<2 là undefined. S1 metric theo đơn vị thời gian đã xác minh, placement trên [0,1]. Không gọi R² là tỷ lệ dự đoán đúng.

In [ ]:
# None: kế thừa cấu hình; sửa giá trị trước khi chạy cell này.
S08_OPTIONS = {
    'decision_mode': None,
    'gpu_ids': None,  # Kế thừa [0, 1]; không đổi khi stage đang chạy.
    'selection_reason': None,
    'selection_approved': None,
    'validation_hash': None
}
run_section("S08", S08_OPTIONS)

Cách đọc kết quả: Metric MAE/RMSE theo units, R² có thể âm và không accuracy. Bảng validation khác bảng final test. Survival vs placement khác đơn vị, không xếp hạng bằng MAE chung. Reload pipeline và batch prediction xác minh trước bàn giao.

Hạn chế và bàn giao: Bảy fit core logic, model/predictions/feature order/backend/cohort metadata mới. RAM/VRAM budget vượt ghi resource_limited; không đổi OLS/CPU/sample ngầm. Prior test exposure unknown phải công bố.

Nếu bảng/hình chưa sinh hoặc stage pending, chưa có kết luận nghiên cứu. Hãy đọc reason, sửa đúng config/source rồi chạy lại S00 và phần này.

Bản5.9: X/y float64 dạng memorymap tách khỏi metadata chuỗi; worker nhận/trả đường dẫn, predict/reload/output theo lô50000. Giữ một Pipeline.fit toàn train, bảy thí nghiệm và G4 trước test. Mmap không biến OLS thành incremental; metrics còn đọc một prediction frame đầy đủ và được tính trong cổng RAM. D5 chưa hoàn tất.

<a id="S09"></a>
## S09. Timing contribution, sai số và giải thích

Câu hỏi kiểm tra: Micro là player-match; match-aware xử lý trọng số match, team-aware placement kiểm actual nhất quán trong đội. Error bins chốt từ train, low-N/coverage rõ. Coefficients theo scaler/đa cộng tuyến, không nhân quả.

Input/provenance: stage dependencies được kiểm từ manifest/checksum; phạm vi lấy từ S00, không lấy latest theo tên file.

Phương pháp và lý do: Đọc saved predictions, exact one-to-one row-ID/target/split trước delta; không dùng giao nhỏ che missing rows. Delta=candidate-reference, MAE/RMSE âm tốt hơn, R² dương tốt hơn.

So sánh:

$$
\Delta M=M_{\mathrm{T1}}-M_{\mathrm{T0}}.
$$

Paired bootstrap lấy cùng tập match IDs có hoàn lại cho hai nhánh, giữ số lần mỗi match xuất hiện; tính lại metric từ contributions, không trung bình RMSE/R² từng match.

Match-aware weights $w_i=1/n_{\mathrm{match}(i)}$, MAE là $\sum_iw_i|e_i|/\sum_iw_i$; R² dùng weighted mean và weighted SST đúng, không trung bình R²/match. Team-aware placement trung bình prediction trong đội sau kiểm target nhất quán.

In [ ]:
# None: kế thừa cấu hình; sửa giá trị trước khi chạy cell này.
S09_OPTIONS = {
    'bootstrap_replicates': None,
    'bootstrap_seed': None
}
run_section("S09", S09_OPTIONS)

Cách đọc kết quả: Residual=actual-predicted. Quan sát mode/miền target nơi sai cao nhưng không tối ưu lại test. CI nếu làm resample match paired có multiplicity, tính lại metrics; thiếu CI không vẽ khoảng/significance.

Hạn chế và bàn giao: Timing comparison/errors/coefficients/uncertainty và source figures; consumer S10. CI không bao gồm mọi bất định model selection hoặc repeated-player dependence.

Nếu bảng/hình chưa sinh hoặc stage pending, chưa có kết luận nghiên cứu. Hãy đọc reason, sửa đúng config/source rồi chạy lại S00 và phần này.

B?n5.10: ki?m ??y ?? row IDs/identity/target tr??c gh?p T0/T1, metrics v? error slices tr?n database ??a; bootstrap gi? RNG/multiplicity c?p tr?n v?i vectors numeric c? guard. M?u v? gi? ??ng RandomState/seed v? ch? ??c ba c?t Parquet theo l?50000; workspace ch? s? O(N) v?n c? guard. Kh?ng l?y trung b?nh RMSE/R2 t?ng l?, kh?ng sample fit/metrics.

Mỗi kết nối database đĩa dùng spill riêng, tránh ghi đè file tạm khi hai kết nối cùng hoạt động; giới hạn SQL không đổi.

<a id="S10"></a>
## S10. Khóa và kiểm tra kết quả

Câu hỏi kiểm tra: Thiếu core/stale/corrupt chặn khóa. Optional not_requested khác completed; không fake model/metric/artifact0. Release snapshot mới bất biến, read-back/checksum trước công bố.

Input/provenance: stage dependencies được kiểm từ manifest/checksum; phạm vi lấy từ S00, không lấy latest theo tên file.

Phương pháp và lý do: Chọn core artifacts theo completed compatible stages và required matrix, không quét mọi CSV/model còn sót làm official. Source/split/cohort/code/config/backend/actual run IDs đều cần provenance.

In [ ]:
# None: kế thừa cấu hình; sửa giá trị trước khi chạy cell này.
S10_OPTIONS = {}
run_section("S10", S10_OPTIONS)

Cách đọc kết quả: Bảng trạng thái/manifest khác số file tồn tại. Development release chỉ kiểm logic, report_ready=false. Khi canonical output ghi đè, release đã khóa phải còn đọc được.

Hạn chế và bàn giao: final_results pointer và release/manifest cụ thể. S11 report_only không cần raw/current checkpoint; chép bản portable với source/dependencies đã chuẩn bị.

Nếu bảng/hình chưa sinh hoặc stage pending, chưa có kết luận nghiên cứu. Hãy đọc reason, sửa đúng config/source rồi chạy lại S00 và phần này.

B?n5.10: inventory bytes/hash v? c? JSON tr??c copy; snapshot staging c?ngfilesystem, checksum/gi?i m?/load l?i tr??c rename; pointer cu?i. L?i copy/validate gi? b?n c? v? d?n staging. L?i pointer sau rename c? th? c?n snapshot ??y ?? ch?a tr? t?i, kh?ng completed. report_only preview nrows25 v? kh?ng collect fullprediction.

<a id="S11"></a>
## S11. Đọc lại kết quả đã khóa trong cùng notebook
Câu hỏi: số liệu đến từ release nào, scope gì, có đủ core và checksum hợp lệ không?

Chế độ report_only bỏ toàn tính toán S01-S10; cell sau chỉ gọi loader một manifest cụ thể. Không mở raw, train, download, chọn model hoặc tạo run mới. Tables/charts được hiển thị theo source của snapshot, không theo current outputs.

Cách đọc: dataset/source/chất lượng trước RQ1, rồi centers/sizes/outcomes RQ2, model metrics hai target, T0/T1, error/coefficient và uncertainty. Nếu release fixture, không dùng số này viết kết quả PUBG chính thức.

Output đang lưu trong bản local ngày 14/10/2026: báo cáo synthetic, scope development, đọc từ quangthinng/project-pubg-abc/versions/3; release ID 23e06692771840c68def99924dc6dd29. Giữ 34 biểu đồ cùng bảng và kiểm đọc lại 408 mục; không phải kết quả PUBG thật. Trong bản tải này, S01-S10 không có output thực thi riêng; báo cáo nằm ở S11. Không tự thêm output hoặc số lượt chạy vào các cell đó.

11 cell hỗ trợ tạm đã được bỏ; 16 cell code chính giữ nguyên. Muốn chạy lại, cấu hình ngay trong cell parameters của S00 rồi chạy bootstrap và section-runner; không dựa vào biến từ cell tạm hoặc RAM phiên cũ. Mặc định chưa cho phép xử lý dữ liệu. Chỉ đọc báo cáo thì dùng report_only và điền root/release/version cụ thể theo README; quyết định K/G4 của fixture không chuyển sang dữ liệu thật.

Ngày 16/10/2026: src đã được sửa cách lưu S01 để giảm bản sao Parquet. Các output nhúng bên dưới vẫn là báo cáo fixture từ Version 3 trước đợt sửa, được giữ nguyên để tham khảo, không phải bằng chứng code mới đã chạy trên Kaggle hoặc kết quả PUBG thật. Kiểm đối chiếu code mới và bản cũ được ghi riêng trong CHANGELOG_FIXES.md; không ép tái sử dụng checkpoint có chữ ký code cũ.

In [ ]:
if EXECUTION_MODE == "report_only":
    summary = display_release(RELEASE_MANIFEST)
elif END_STAGE == "S10":
    pointer = read_json(ctx.path("manifests/final_results.json"))
    RELEASE_MANIFEST = str(ctx.root/pointer["manifest"])
    summary = display_release(RELEASE_MANIFEST)
else:
    print("Chưa tới finalization; chỉ đọc diagnostics/stage đã chạy, chưa có final summary")

<a id="S12"></a>
## S12. Kết luận khoa học, giới hạn và bàn giao
Mỗi phát biểu trong báo cáo phải dẫn bảng/hình/run: trên population/scope/mode nào, N bao nhiêu, hệ số/metric/delta thực đo gì và giới hạn gì. Không viết trước rằng timing cải thiện hoặc cụm nào tốt nhất.

Giới hạn bắt buộc: hồi cứu sau trận, dữ liệu quan sát, tên người chơi không ID bất biến, proxy duration, mismatch event/aggregate, phase-survival coupling, các đồng đội cùng outcome, chronology/selection/test exposure và giới hạn tài nguyên. CI theo match chưa giải quyết mọi phụ thuộc repeated players.

Liên hệ môn học: C1 quy trình KDD; C2 thống kê/cleaning/scaling; C6 hồi quy/residual/đa cộng tuyến; C7 KMeans/K/stability; C4 nguyên tắc đánh giá. C3/C5 không ép vào target liên tục.

Bộ nộp: một notebook có output, code/config/requirements/README, tables/figures và manifest cụ thể; báo cáo/slides cùng release. Demo report_only, không train raw lớn trong giờ bảo vệ. Các thành viên luyện giải thích per_mode/scaling/K, split group match, leakage, baseline, MAE/RMSE/R² và T0/T1.

Bàn giao: ghi root/project_id/code-config version, scope, stage cuối đã completed, pending reason, artifact paths và checksum, writer và section tiếp theo. File có tên hoặc cell cuối xanh không chứng minh toàn quy trình hoàn tất.

In [ ]:
if EXECUTION_MODE != "report_only":
    display(pd.DataFrame([{"stage":s,"status":r.get("status"),"reason":r.get("reason",""),
                          "outputs":len(r.get("artifacts",[])),"writer":r.get("writer")}
                         for s,r in ctx.manifest().items()]))
    print("Runtime outputs (chưa chứng minh Kaggle đã lưu):",ctx.out)
    print("Manifest bàn giao:",ctx.manifest_path)
    print("Notebook cần lưu đúng tên PUBG_COURSEWORK.ipynb trên local và cùng Kaggle notebook identity; không thêm (1)/(2).")
else:
    print("Đang đọc:",RELEASE_MANIFEST)
    print("Chỉ đọc release. Không có stage, model hoặc run mới được tạo.")

### Kiểm gói đã lưu trước bàn giao
Save Version với output, sau đó Add Input gắn output của đúng owner/notebook/versions/NUMBER. Điền SAVED_OUTPUT_ROOT và SAVED_VERSION rồi chạy cell sau. Không lấy working runtime đang sống làm bằng chứng Kaggle đã lưu. Hàm không đăng nhập/API, version identity do người dùng cung cấp; checksum kiểm nội dung, không xác minh quyền tài khoản. Chỉ dùng gói code/model của nhóm đáng tin cậy.
Resume: phiên mới điền RESUME_INPUT, RESUME_VERSION và SOURCE_DIRECTORY; working project phải trống/mới. report_only đọc release Input cụ thể, không restore/copy/write. Lưu notebook về local bằng cùng tên, không tạo (1)/(2).

In [ ]:
if SAVED_OUTPUT_ROOT and SAVED_VERSION:
    handoff_scope = summary["scope"] if EXECUTION_MODE=="report_only" else EXECUTION_MODE
    _, readback = verify_handoff(SAVED_OUTPUT_ROOT, handoff_scope, SAVED_VERSION)
    display(pd.DataFrame([readback]))
else:
    print("Chưa xác minh Kaggle Version đã lưu; kiểm Output và gắn version cụ thể trước bàn giao.")